# 🛡️ Governance-Focused Data Preprocessing with IBM's Data Prep Kit

This notebook demonstrates how to build a governance-first data pipeline using IBM's open-source Data Prep Kit (DPK). The pipeline focuses on ensuring safety, privacy, and quality in datasets used for AI training.


## 🔍 Why Governance Matters in AI Training

Responsible AI begins with responsible data. Governance in data preprocessing ensures that training datasets are:
- Free from toxic or harmful content
- Compliant with privacy regulations
- High-quality and diverse
- Traceable and auditable


## 🧰 Key Transforms in This Recipe

We will use the following transforms from DPK:

- `HAPTransform`: Detects hate, abuse, and profanity.
- `PIIRedactorTransform`: Identifies and redacts personally identifiable information.
- `DocQualityTransform`: Scores documents based on structure and coherence.


## Prerequisites


#### <b>1- Setup a virtual environment for running the notebook. The instructions below provide one example using python venv. </b>

```
python -m venv venv
source venv/bin/activate
pip install jupyterlab

```

#### <b>2- The HAP model used is stored in HuggingFace. Users need to provide their own read-access token for retrieving the models. In this notebook, we assume the users have used an environment variable named HF_READ_ACCESS_TOKEN for storing their HF token. </b>

```
export HF_READ_ACCESS_TOKEN='hf_xxx'
```

## Install DPK library to environment

In [ ]:
!pip install --no-cache "data-prep-toolkit-transforms[hap, pii_redactor, doc_quality, filter]==1.1.2.post1"

In [ ]:
!pip install --no-cache "data-prep-toolkit==1.0.0.post1"

## 🔗 Building a Governance Pipeline with TransformsChain

DPK provides a `TransformsChain` API to compose multiple transforms into a single pipeline. This makes it easy to enforce governance policies in a structured and scalable way.


In [ ]:
import os
import pathlib
import pyarrow.parquet as pq
from dpk_hap import HAPTransform
from dpk_pii_redactor import PIIRedactorTransform
from dpk_doc_quality import DocQualityTransform
from data_processing.data_access import DataAccessLocal
from dpk_transform_chain import TransformsChain

In [ ]:
%env HF_READ_ACCESS_TOKEN=ENTER_YOUR_HF_TOKEN

### Set up each Transform's configuration parameters

In [ ]:
base = "./test-data/"

hap_params = {"model_name_or_path": "ibm-granite/granite-guardian-hap-38m",
              "annotation_column": "hap_score",
              "doc_text_column": "contents",
              "inference_engine": "CPU",
              "max_length": 512,
              "batch_size": 128,
             }

pii_params = {"entities": ["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER"],
              "operator": "replace",
              "transformed_contents": "title",
             }


dq_params = {"doc_content_column": "contents",
             "bad_word_filepath": os.path.join(base, "ldnoobw", "en"),
             "text_lang": "en",
            }

### Define Data Access to connect dataset to be processed

In [ ]:
da_config = {
            "config": {
                "input_folder": os.path.join(base, 'input'),
                "output_folder": os.path.join(base, 'output'),
            },
            "files_to_use": [".parquet"]
        }

data_access = DataAccessLocal(**da_config)

### Set up Transform Chain orchestrator and run

In [ ]:
orch = TransformsChain(data_access=data_access,
                       transforms=[
                                   HAPTransform(hap_params),
                                   PIIRedactorTransform(pii_params),
                                   DocQualityTransform(dq_params),
                                  ]
                      )


In [ ]:
orch.run()

### Review output - with `TransformsChain`, only 1 output file will be produced for each input file, as opposed to each transform producing it's own outputfile 

#### Annotations: Output should include new columns with annotations provided by each transform: 
- detected_pii - shows a list of the type of PII (if any) detected from a row
- title - redacted PII contents
- hap_score - a score from 0 to 1 showing the likelyhood the contents of a row contains HAP
- docq_total_words - the total number of words
- docq_mean_word_len - the mean of words' lengths
- docq_symbol_to_word_ratio - the ratio of symbol-to-word ratio
- docq_sentence_count - the number of sentences
- docq_curly_bracket_ratio - the ratio between the number of occurrences of { or } over the text length
- docq_lorem_ipsum_ratio - the ratio between the number of occurrences of lorem ipsum over the text length. Lorem ipsum, or lipsum as it is sometimes known, is dummy text used in laying out print, graphic or web designs.
- docq_contain_bad_word - whether text containst bad words
- docq_bullet_point_ratio - the ratio of lines starting with a bullet point
- docq_ellipsis_line_ratio - the ratio of lines ending with an ellipsis
- docq_alphabet_word_ratio - the ratio of words having at least one alphabetic character
- docq_contain_common_en_words - whether the given text contains common English words like the, and, to, that, of, with, be, and have


In [ ]:
import glob
output = glob.glob(os.path.join(data_access.get_output_folder(), '*'))

In [ ]:
import pandas as pd
pd.read_parquet(output[0], engine='pyarrow')

## ✅ Summary

This notebook demonstrated how to build a governance-focused data preprocessing pipeline using IBM's Data Prep Kit. By chaining together key transforms, you can ensure your AI training data is safe, compliant, and high-quality.
